# 00a - Sentinel-2 L2A Time-Series Downloader & Kaggle Packager

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook queries **Element84 Earth Search STAC API** (Open S3 COG Endpoint, zero SAS token / 403 authorization requirement) for cloud-free Sentinel-2 L2A scenes over Liwa Oasis (`[53.5, 22.8, 54.2, 23.3]`), downloads multispectral bands into `/kaggle/working/`, and packages it as an isolated Kaggle Dataset (`assemelqersh/sentinel2-liwa-l2a`).

The mechanistic justification for this step is: Utilizing Element84 Earth Search STAC completely eliminates 403 Forbidden HTTP errors associated with SAS token signing on Microsoft Planetary Computer while guaranteeing open AWS S3 COG band access.

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (pystac-client, pystac, rasterio, geopandas, folium, requests) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("pystac_client", "pystac-client")
install_if_missing("pystac", "pystac")
install_if_missing("rasterio", "rasterio")
install_if_missing("geopandas", "geopandas")
install_if_missing("folium", "folium")

# Step 1: Environment Setup & AOI Definition
# The mechanistic justification for this step is: Defining a standardized WGS84 GeoJSON bounding box ([53.5, 22.8, 54.2, 23.3]) isolates the agricultural fields of Liwa Oasis, preventing unnecessary spatial query overhead during STAC catalog searches.
import os
import sys
import json
import requests
from tqdm import tqdm
import geopandas as gpd
from shapely.geometry import box
from pystac_client import Client

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "../data/processed/sentinel2-liwa-l2a"
os.makedirs(OUT_DIR, exist_ok=True)

LIWA_BBOX = [53.5, 22.8, 54.2, 23.3]
LIWA_AOI = box(*LIWA_BBOX)
print(f"Sentinel-2 Downloader initialized. Output directory: {OUT_DIR}")


In [ ]:
# Step 2: Query Element84 Earth Search STAC for Open S3 Sentinel-2 L2A Scenes
# The mechanistic justification for this step is: Element84 STAC provides unauthenticated public S3 COG access for bands B02, B03, B04, B08, B11, B12, SCL, bypassing SAS token expiration.
stac_endpoints = [
    "https://earth-search.aws.element84.com/v1",
    "https://planetarycomputer.microsoft.com/api/stac/v1"
]

s2_items = []
for endpoint in stac_endpoints:
    try:
        print(f"Connecting to STAC endpoint: {endpoint}...")
        catalog = Client.open(endpoint)
        search = catalog.search(
            collections=["sentinel-2-l2a"],
            bbox=LIWA_BBOX,
            datetime="2024-10-01/2025-09-30",
            query={"eo:cloud_cover": {"lt": 10}},
            max_items=4
        )
        items = list(search.items())
        if items:
            s2_items = items
            print(f"Successfully found {len(s2_items)} cloud-free Sentinel-2 L2A scenes via {endpoint}.")
            break
    except Exception as e:
        print(f"Endpoint {endpoint} note: {e}")

band_key_map = {
    "B02": ["blue", "B02"],
    "B03": ["green", "B03"],
    "B04": ["red", "B04"],
    "B08": ["nir", "B08"],
    "B11": ["swir16", "B11"],
    "B12": ["swir22", "B12"],
    "SCL": ["scl", "SCL"]
}

for item in s2_items:
    date_str = item.datetime.strftime("%Y%m%d")
    scene_dir = os.path.join(OUT_DIR, f"S2_L2A_{date_str}_{item.id[:20]}")
    os.makedirs(scene_dir, exist_ok=True)
    
    for band_name, posibles in band_key_map.items():
        asset_url = None
        for p in posibles:
            if p in item.assets:
                asset_url = item.assets[p].href
                break
        
        if asset_url:
            save_path = os.path.join(scene_dir, f"{band_name}.tif")
            if not os.path.exists(save_path):
                res = requests.get(asset_url, stream=True)
                if res.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in res.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
                    mb = os.path.getsize(save_path) / (1024 * 1024)
                    print(f"Saved {save_path} ({mb:.2f} MB)")

s2_meta = {
    "title": "Sentinel-2 L2A Liwa Oasis Time-Series Dataset",
    "id": "assemelqersh/sentinel2-liwa-l2a",
    "licenses": [{"name": "CC-BY-4.0"}]
}
with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(s2_meta, f, indent=2)

print(f"[SUCCESS] Sentinel-2 Dataset packaged cleanly in {OUT_DIR}")
